Nettoyage des données Airbnb pour plusieurs villes (Paris, Bordeaux, Pays Basque).

In [ ]:

import os
import pandas as pd
from datetime import datetime

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

DATA_DIR = "."
OUTPUT_DIR = "./cleaned_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def exists(path):
    return os.path.exists(path)


In [ ]:

def to_float_price(series):
    if series is None:
        return series
    cleaned = series.astype(str).str.replace(r"[^\d\.,-]", "", regex=True)
    cleaned = cleaned.str.replace(",", ".", regex=False)
    cleaned = cleaned.str.replace(r"(?<=\d)\.(?=\d{3}(\D|$))", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

def normalize_text(s):
    if s is None:
        return s
    return s.astype(str).str.strip().str.lower()

def parse_date_safe(s):
    if s is None:
        return s
    return pd.to_datetime(s, errors="coerce", infer_datetime_format=True)

def clean_listings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'id' in df.columns:
        df = df.drop_duplicates(subset=['id'])
    else:
        df = df.drop_duplicates()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip()
    if 'price' in df.columns:
        df['price'] = to_float_price(df['price'])
    for col in ['minimum_nights','number_of_reviews','calculated_host_listings_count','availability_365','number_of_reviews_ltm']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ['last_review']:
        if col in df.columns:
            df[col] = parse_date_safe(df[col])
    for col in ['neighbourhood','neighbourhood_group','neighbourhood_cleansed']:
        if col in df.columns:
            df[col] = normalize_text(df[col])
    return df

def build_review_stats(reviews: pd.DataFrame) -> pd.DataFrame:
    reviews = reviews.copy()
    if 'listing_id' not in reviews.columns and 'listing' in reviews.columns:
        reviews = reviews.rename(columns={'listing': 'listing_id'})
    for col in reviews.select_dtypes(include=['object']).columns:
        reviews[col] = reviews[col].astype(str).str.strip()
    if 'date' in reviews.columns:
        reviews['date'] = parse_date_safe(reviews['date'])
    cols = reviews.columns
    if 'listing_id' in cols and 'id' in cols:
        agg = (reviews.groupby('listing_id', as_index=False)
               .agg(review_count=('id','count'), last_review=('date','max')))
        return agg
    elif 'listing_id' in cols:
        agg = (reviews.groupby('listing_id', as_index=False)
               .agg(review_count=('listing_id','count'), last_review=('date','max')))
        return agg
    else:
        return pd.DataFrame(columns=['listing_id','review_count','last_review'])

def safe_merge_listings_neigh(listings: pd.DataFrame, neigh: pd.DataFrame) -> pd.DataFrame:
    if neigh is None or neigh.empty:
        return listings.copy()
    cand_keys = [('neighbourhood','neighbourhood'),
                 ('neighbourhood_group','neighbourhood_group'),
                 ('neighbourhood','neighbourhood_cleansed'),
                 ('neighbourhood_cleansed','neighbourhood')]
    for df in (listings, neigh):
        for c in df.columns:
            if df[c].dtype == 'object':
                df[c] = df[c].astype(str).str.strip().str.lower()
    for lk, rk in cand_keys:
        if lk in listings.columns and rk in neigh.columns:
            return listings.merge(neigh, left_on=lk, right_on=rk, how='left')
    return listings.copy()


In [ ]:

def load_city_files(city_prefix: str):
    listings_path = os.path.join(DATA_DIR, f"{city_prefix}_listings.csv")
    neigh_csv_path = os.path.join(DATA_DIR, f"{city_prefix}_neighbourhoods.csv")
    reviews_path = os.path.join(DATA_DIR, f"{city_prefix}_reviews.csv")

    if not exists(listings_path):
        raise FileNotFoundError(f"Fichier listings introuvable : {listings_path}")

    listings = pd.read_csv(listings_path)
    neigh = pd.read_csv(neigh_csv_path) if exists(neigh_csv_path) else None
    reviews = pd.read_csv(reviews_path) if exists(reviews_path) else None
    return listings, neigh, reviews

def process_city(prefix: str) -> pd.DataFrame:
    print(f"\n==== {prefix.upper()} ====")
    listings, neigh, reviews = load_city_files(prefix)
    print("Dimensions brutes listings:", listings.shape)
    listings = clean_listings(listings)
    print("Après nettoyage listings:", listings.shape)
    merged = safe_merge_listings_neigh(listings, neigh if neigh is not None else pd.DataFrame())
    print("Après fusion avec neighbourhoods.csv:", merged.shape)
    if reviews is not None and not reviews.empty:
        stats = build_review_stats(reviews)
        merged = merged.merge(stats, left_on='id', right_on='listing_id', how='left')
        merged = merged.drop(columns=['listing_id'], errors='ignore')
        print("Après ajout stats reviews:", merged.shape)
    else:
        print("Aucun reviews.csv trouvé pour", prefix)
    out_path = os.path.join(OUTPUT_DIR, f"cleaned_{prefix}.csv")
    merged.to_csv(out_path, index=False)
    print("Sauvegardé:", out_path)
    return merged


In [ ]:

cities = ['paris', 'bordeau', 'paysbasque']
cleaned = []
for city in cities:
    try:
        dfc = process_city(city)
        cleaned.append(dfc.assign(city=city))
    except FileNotFoundError as e:
        print(e)
if cleaned:
    combined = pd.concat(cleaned, ignore_index=True, sort=False)
    combined_path = os.path.join(OUTPUT_DIR, "cleaned_all_cities.csv")
    combined.to_csv(combined_path, index=False)
    print("Fichier combiné sauvegardé:", combined_path)
    display(combined.head(10))
else:
    print("Aucune ville traitée.")


In [ ]:

combined_path = os.path.join(OUTPUT_DIR, "cleaned_all_cities.csv")
if exists(combined_path):
    df = pd.read_csv(combined_path)
    print("Lignes, Colonnes:", df.shape)
    display(df.head(5))
    if 'price' in df.columns:
        print(df['price'].describe())
else:
    print("Le fichier combiné n'existe pas encore.")
